# Crezee 2022 map and shapefile outline

This Colab notebook mounts Google Drive, locates the Crezee 2022 GeoTIFF, displays the classified raster, and exports a dissolved outline of the **open-water and peat-forest classes (1, 4, and 5)** as a zipped ESRI Shapefile. Other classes and NoData pixels are excluded.

## 1. Install and import the geospatial packages

In [ ]:
!pip -q install rasterio geopandas matplotlib

In [ ]:
from pathlib import Path
import shutil

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from google.colab import drive, files
from rasterio.enums import Resampling
from rasterio.features import shapes
from rasterio.warp import calculate_default_transform, reproject
from shapely.geometry import shape
from shapely.ops import unary_union

## 2. Mount Drive and find the GeoTIFF

The path below matches `My Drive > Colab Notebooks > PanAfrica_LU` in the supplied screenshot. The wildcard tolerates small filename changes after `Crezee_2022`.

In [ ]:
drive.mount('/content/drive')

data_dir = Path('/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU')
matches = sorted(data_dir.glob('Crezee_2022*.tif'))

if not matches:
    raise FileNotFoundError(
        f'No Crezee_2022*.tif found in {data_dir}. '
        'Check the folder and filename in Google Drive.'
    )
if len(matches) > 1:
    print('Multiple matches found; using the first one:')
    for candidate in matches:
        print(f'  - {candidate.name}')

tif_path = matches[0]
print(f'Using: {tif_path}')

## 3. Inspect and display the map

Only a downsampled display array is loaded for the map, which keeps an approximately 860 MB raster manageable in Colab. The later classification and polygonization steps process the original raster in blocks.

In [ ]:
max_display_side = 2000

with rasterio.open(tif_path) as src:
    if src.crs is None:
        raise ValueError('The GeoTIFF has no CRS; a georeferenced shapefile cannot be created safely.')

    scale = min(1.0, max_display_side / max(src.width, src.height))
    display_width = max(1, round(src.width * scale))
    display_height = max(1, round(src.height * scale))
    raster = src.read(
        1,
        out_shape=(display_height, display_width),
        resampling=Resampling.nearest,
        masked=True,
    )
    bounds = src.bounds
    raster_crs = src.crs
    raster_shape = (src.height, src.width)
    nodata = src.nodata

print(f'Raster size: {raster_shape[1]:,} x {raster_shape[0]:,} pixels')
print(f'CRS: {raster_crs}')
print(f'Bounds: {bounds}')
print(f'NoData: {nodata}')

fig, ax = plt.subplots(figsize=(14, 10))
image = ax.imshow(
    raster,
    extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
    origin='upper',
    interpolation='nearest',
    cmap='tab20',
)
ax.set_title(tif_path.name)
ax.set_xlabel('Easting / longitude')
ax.set_ylabel('Northing / latitude')
ax.set_aspect('equal')
fig.colorbar(image, ax=ax, shrink=0.75, label='Class value')
plt.show()

## 4. Create the open-water and peat-forest outline

The selected land-cover values are classes 1, 4, and 5. The cell mask is first aggregated to a 1 km grid in the EPSG:6933 equal-area projection. A coarse cell is selected if **any** contributing source pixel belongs to one of those classes, retaining small features while making polygonization much faster.

In [ ]:
selected_classes = (1, 4, 5)  # Open water and peat forest
grid_size_m = 1000
target_crs = 'EPSG:6933'  # Equal-area projection with metre units
source_mask_path = Path('/content/selected_classes_mask.tif')
print(f'Classes selected for export: {selected_classes}')

# Write a compact 0/1 mask block by block, avoiding a full-resolution array in RAM.
with rasterio.open(tif_path) as src:
    mask_profile = src.profile.copy()
    mask_profile.update(count=1, dtype='uint8', nodata=0, compress='deflate')
    with rasterio.open(source_mask_path, 'w', **mask_profile) as mask_dst:
        for _, window in src.block_windows(1):
            block = src.read(1, window=window, masked=True)
            selected = (~np.ma.getmaskarray(block)) & np.isin(block.data, selected_classes)
            mask_dst.write(selected.astype('uint8'), 1, window=window)

# Reproject and aggregate to 1 km. Resampling.max means 'any selected pixel'.
with rasterio.open(source_mask_path) as mask_src:
    coarse_transform, coarse_width, coarse_height = calculate_default_transform(
        mask_src.crs, target_crs, mask_src.width, mask_src.height,
        *mask_src.bounds, resolution=grid_size_m,
    )
    coarse_mask = np.zeros((coarse_height, coarse_width), dtype='uint8')
    reproject(
        source=rasterio.band(mask_src, 1),
        destination=coarse_mask,
        src_transform=mask_src.transform,
        src_crs=mask_src.crs,
        dst_transform=coarse_transform,
        dst_crs=target_crs,
        src_nodata=0,
        dst_nodata=0,
        resampling=Resampling.max,
    )

selected_cell_count = int(coarse_mask.sum())
print(f'Coarse grid: {coarse_width:,} × {coarse_height:,} cells')
print(f'Selected 1 km cells: {selected_cell_count:,}')
if selected_cell_count == 0:
    raise ValueError(f'No valid pixels belonging to classes {selected_classes} were found.')

polygons = [
    shape(geometry)
    for geometry, value in shapes(
        coarse_mask, mask=coarse_mask.astype(bool),
        transform=coarse_transform, connectivity=8,
    )
    if value == 1
]
selected_geometry = unary_union(polygons)
outline = gpd.GeoDataFrame(
    {
        'name': ['water_peat'],
        'classes': ['1,4,5'],
        'grid_m': [grid_size_m],
    },
    geometry=[selected_geometry],
    crs=target_crs,
)

ax = outline.plot(figsize=(12, 8), facecolor='none', edgecolor='red', linewidth=0.5)
ax.set_title('Open-water and peat-forest areas (classes 1, 4, and 5)')
ax.set_xlabel('Easting / longitude')
ax.set_ylabel('Northing / latitude')
ax.set_aspect('equal')
plt.show()

outline

## 5. Export and download the shapefile

A shapefile consists of several companion files, so they are bundled into one ZIP for download.

In [ ]:
output_dir = Path('/content/Crezee_2022_water_peat')
output_dir.mkdir(parents=True, exist_ok=True)
shapefile_path = output_dir / 'Crezee_2022_water_peat.shp'

outline.to_file(shapefile_path, driver='ESRI Shapefile', index=False)
zip_path = Path(shutil.make_archive('/content/Crezee_2022_water_peat', 'zip', output_dir))

print(f'Created: {zip_path}')
files.download(str(zip_path))